# 02. Feature Engineering & Feature Store
Aplicaremos funções de janela (Window) do Spark para criar inteligência temporal e salvar na Feature Store.

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import count, sum, avg, max, datediff, col, when, unix_timestamp

# O uso da lib FeatureEngineeringClient é a melhor prática atual no Databricks
try:
    from databricks.feature_engineering import FeatureEngineeringClient
    fe = FeatureEngineeringClient()
    HAS_FE = True
except ImportError:
    print("Lib feature_engineering não encontrada. Rodando modo local.")
    HAS_FE = False

ENVIRONMENT = "databricks_volume"
BASE_PATH = "/Volumes/workspace/default/raw_data" if ENVIRONMENT == "databricks_volume" else "file:///tmp/mlops"
raw_mlops_path = f"{BASE_PATH}/mlops/raw_call_logs"


In [ ]:
df_logs = spark.read.parquet(raw_mlops_path).withColumn("call_time_sec", unix_timestamp("call_date"))

# Criando Window Functions Particionadas por Cliente e Ordenadas por Tempo
w_30d = Window.partitionBy("customer_id").orderBy("call_time_sec").rangeBetween(-30 * 86400, 0)
w_7d = Window.partitionBy("customer_id").orderBy("call_time_sec").rangeBetween(-7 * 86400, 0)

print("Calculando variáveis temporais e métricas de reincidência...")
df_features = df_logs \
    .withColumn("calls_last_7d", count("call_id").over(w_7d)) \
    .withColumn("calls_last_30d", count("call_id").over(w_30d)) \
    .withColumn("avg_wait_time_30d", avg("queue_wait_time").over(w_30d)) \
    .withColumn("sum_transfers_30d", sum("transfer_count").over(w_30d))

# Feature de SLA estourado: teve chamada pendente nos últimos 7 dias?
df_features = df_features.withColumn(
    "has_unresolved_7d",
    max(when(col("resolution_status") == "Pendente", 1).otherwise(0)).over(w_7d)
)

# Tabela Gold: Para o modelo, precisamos de uma linha por cliente (Agrupamento Final)
# Vamos pegar apenas os clientes do último mês para treino.
df_gold = df_features.groupBy("customer_id").agg(
    max("calls_last_7d").alias("max_calls_last_7d"),
    max("avg_wait_time_30d").alias("max_avg_wait_time_30d"),
    max("sum_transfers_30d").alias("total_transfers_30d"),
    max("has_unresolved_7d").alias("unresolved_issue_recently"),
    max("bacen_complaint").alias("target_bacen") # O Target (Y)
)

df_gold.show(5)

In [ ]:
if HAS_FE:
    print("Registrando no Databricks Feature Store...")
    # Em um ambiente com Unity Catalog real, isso criaria a tabela no Hive/UC.
    try:
        fe.create_table(
            name="workspace.bacen_mlops.call_center_features",
            primary_keys=["customer_id"],
            df=df_gold,
            description="Features de reincidência de call center para modelo de BACEN"
        )
    except Exception as e:
        print(f"Nota: Tabela pode já existir ou erro de permissão. Detalhe: {e}")
else:
    print("Salvando Feature Table localmente como Delta/Parquet...")
    df_gold.write.mode("overwrite").parquet(f"{BASE_PATH}/mlops/feature_store")
